In [1]:
from lab.sinus_vs_square_hard.data import create_data

nb_points_per_period = 8
nb_periods = 300
nb_points = nb_periods * nb_points_per_period
batch_size = 10 * nb_points_per_period
nb_batches = nb_points // batch_size

dataset_name = f"base_params_{nb_periods}_periods_affine"
X, Y = create_data(nb_periods)
X_batches = X.reshape(nb_batches, batch_size)
Y_batches = Y.reshape(nb_batches, batch_size, 1)


In [2]:
from quantum_simulation.configs import jpc_config_dudas, quantum_parameters_dudas, base_encoding

jpc_config = jpc_config_dudas
quantum_parameters = quantum_parameters_dudas
encoding = base_encoding


In [3]:
from quantum_simulation import Simulator

simulator = Simulator(jpc_config=jpc_config,
                      encoding=encoding)


In [4]:
from quantum_learn import build_f

build_f_quad = build_f.BuildFQuadratures(jpc_config=jpc_config_dudas)
build_f_quad_poly = build_f.BuildFQuadraturesPolynomials(jpc_config=jpc_config_dudas)
build_f_probas = build_f.BuildFPhotonDistribution(jpc_config=jpc_config_dudas, clip_probas=2)



In [5]:
import numpy as np

F_quad = np.empty((nb_batches, batch_size, build_f_quad.output_dim))
F_quad_poly = np.empty((nb_batches, batch_size, build_f_quad_poly.output_dim))
F_probas = np.empty((nb_batches, batch_size, build_f_probas.output_dim))
F_quad.shape

(30, 80, 40)

In [6]:
for i, X in enumerate(X_batches):
    results = simulator.run_simulation(X=X, quantum_parameters_list=[quantum_parameters])[0]
    F_quad[i, :, :] = build_f_quad(results)
    F_quad_poly[i, :, :] = build_f_quad_poly(results)
    F_probas[i, :, :] = build_f_probas(results)


|██████████| 100.0% ◆ elapsed 02m43s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 02m59s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 02m59s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m07s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m15s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m20s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m10s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m10s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m05s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 02m59s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m02s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m00s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m01s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m16s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 04m02s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m26s ◆ remaining 0.00ms  
|██████████| 100.0% ◆ elapsed 03m08s ◆ remaining 0.00ms  
|██████████| 1

In [7]:
X_batches.shape, Y_batches.shape

((30, 80), (30, 80, 1))

In [8]:
import numpy as np
from pathlib import Path

save_dir = Path("datasets") / dataset_name

save_dir.mkdir(parents=True, exist_ok=True)

jpc_config.save(save_dir / "jpc_config.txt")
quantum_parameters.save(save_dir / "quantum_parameters.txt")
encoding.save(save_dir / "encoding.txt")

np.save(save_dir / "X.npy", X_batches)
np.save(save_dir / "Y.npy", Y_batches)
np.save(save_dir / "f_quad.npy", F_quad)
np.save(save_dir / "f_quad_poly.npy", F_quad_poly)
np.save(save_dir / "f_probas.npy", F_probas)


